# Physics-Informed PPO Formulations: Current Results

Generated on 2026-06-17 from the current experiment artifacts in this repository.

Training outputs: `C:\Users\aha173\OneDrive - American University of Beirut\research\TeleopProject\TeleopDocs\TeleopWithRL\matlab_literal_env\policy_gradient_experiments\results\dyn\physics_informed_formulations_01`  
Validation outputs: `C:\Users\aha173\OneDrive - American University of Beirut\research\TeleopProject\TeleopDocs\TeleopWithRL\matlab_literal_env\policy_gradient_experiments\results\dyn\validation_test_suite_01`

This notebook is meant to be both a snapshot and a reusable analysis notebook. The markdown tables below freeze the current numbers, while the code cells can be rerun to reload the CSVs and regenerate the displayed tables.


## Transparency Definitions

Two transparency quantities are tracked because they answer different questions:

- Stable validation/reward metric: $e_T(t) = F_e(t)v_m(t) - F_h(t)v_s(t)$, reported as RMSE in watts. This avoids the singularities that appear when velocities are near zero.
- Actual force/velocity ratio diagnostic: $T_{ratio}(t) = (F_h(t)/v_m(t)) / (F_e(t)/v_s(t))$. This is the requested `f/v` over `f/v` term, but it can explode near zero velocity, so it is plotted and summarized as a diagnostic rather than used alone for ranking.

Control effort is monitored using RMS control voltage $u$, and smoothness is monitored using mean absolute $\Delta u$.


## Formulations

The physics-informed runs are the seven PPO formulations from the current study. F0 keeps the baseline observation and reward; F1-F3 add tracking error derivatives; F4-F5 add acceleration observations; F6 returns to the baseline state and adds the smoothness term.

| Key | Label | Obs Dim | State Features | Reward Terms | Train RMS err mm | RMS u V | Mean abs(delta u) V | Completed |
|---|---|---|---|---|---|---|---|---|
| F0_baseline | Baseline | 5 | x_m x_s v_m v_s u_v | tracking_base control_effort | 17.891 | 0.0694 | 0.0019 | 1.00 |
| F1_error_state_reward | Add Error | 6 | x_m x_s v_m v_s u_v tracking_error | tracking_base control_effort physics_error | 4.911 | 0.1382 | 0.0078 | 1.00 |
| F2_error_dot_state_reward | Add Error Dot | 7 | x_m x_s v_m v_s u_v tracking_error velocity_error | tracking_base control_effort physics_error physics_error_dot | 4.876 | 0.1473 | 0.0137 | 1.00 |
| F3_error_ddot_state_reward | Add Error DDot | 8 | x_m x_s v_m v_s u_v tracking_error velocity_error acceleration_error | tracking_base control_effort physics_error physics_error_dot physics_error_ddot | 6.211 | 0.0929 | 0.0042 | 1.00 |
| F4_accel_state | Accel State | 7 | x_m x_s v_m v_s u_v x_m_ddot x_s_ddot | tracking_base control_effort | 17.958 | 0.0882 | 0.0024 | 1.00 |
| F5_accel_state_reward | Accel State + Reward | 7 | x_m x_s v_m v_s u_v x_m_ddot x_s_ddot | tracking_base control_effort acceleration_error | 15.443 | 0.0887 | 0.0033 | 1.00 |
| F6_effort_plus_delta_u | Effort + Delta U | 5 | x_m x_s v_m v_s u_v | tracking_base control_effort smooth_delta_u | 17.898 | 0.0694 | 0.0019 | 1.00 |

In [ ]:
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, Markdown, display


def find_repo_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    for path in [start, *start.parents]:
        if (path / "matlab_literal_env").exists() and (path / "notebooks").exists():
            return path
    raise RuntimeError("Could not find the TeleopWithRL repository root from the current working directory.")


ROOT = find_repo_root()
PHYSICS_DIR = ROOT / "matlab_literal_env" / "policy_gradient_experiments" / "results" / "dyn" / "physics_informed_formulations_01"
VALIDATION_DIR = ROOT / "matlab_literal_env" / "policy_gradient_experiments" / "results" / "dyn" / "validation_test_suite_01"

physics_summary = pd.read_csv(PHYSICS_DIR / "summary.csv")
validation_summary = pd.read_csv(VALIDATION_DIR / "validation_summary.csv")
validation_group = pd.read_csv(VALIDATION_DIR / "validation_group_summary.csv")
validation_bode = pd.read_csv(VALIDATION_DIR / "validation_bode_all.csv")

print(f"Physics results: {PHYSICS_DIR}")
print(f"Validation results: {VALIDATION_DIR}")


## Training Snapshot

The physics-informed training comparison below comes directly from `physics_informed_formulations_01/summary.csv`. These are rollout/training-study metrics, not the full validation-suite metrics.

| Key | Label | Obs Dim | State Features | Reward Terms | Train RMS err mm | RMS u V | Mean abs(delta u) V | Completed |
|---|---|---|---|---|---|---|---|---|
| F0_baseline | Baseline | 5 | x_m x_s v_m v_s u_v | tracking_base control_effort | 17.891 | 0.0694 | 0.0019 | 1.00 |
| F1_error_state_reward | Add Error | 6 | x_m x_s v_m v_s u_v tracking_error | tracking_base control_effort physics_error | 4.911 | 0.1382 | 0.0078 | 1.00 |
| F2_error_dot_state_reward | Add Error Dot | 7 | x_m x_s v_m v_s u_v tracking_error velocity_error | tracking_base control_effort physics_error physics_error_dot | 4.876 | 0.1473 | 0.0137 | 1.00 |
| F3_error_ddot_state_reward | Add Error DDot | 8 | x_m x_s v_m v_s u_v tracking_error velocity_error acceleration_error | tracking_base control_effort physics_error physics_error_dot physics_error_ddot | 6.211 | 0.0929 | 0.0042 | 1.00 |
| F4_accel_state | Accel State | 7 | x_m x_s v_m v_s u_v x_m_ddot x_s_ddot | tracking_base control_effort | 17.958 | 0.0882 | 0.0024 | 1.00 |
| F5_accel_state_reward | Accel State + Reward | 7 | x_m x_s v_m v_s u_v x_m_ddot x_s_ddot | tracking_base control_effort acceleration_error | 15.443 | 0.0887 | 0.0033 | 1.00 |
| F6_effort_plus_delta_u | Effort + Delta U | 5 | x_m x_s v_m v_s u_v | tracking_base control_effort smooth_delta_u | 17.898 | 0.0694 | 0.0019 | 1.00 |

In [ ]:
training_view = physics_summary.assign(
    tracking_rmse_mm=physics_summary["tracking_rmse_m"] * 1000.0,
)[[
    "key",
    "label",
    "obs_dim",
    "state_features",
    "reward_terms",
    "tracking_rmse_mm",
    "rms_u_v",
    "mean_abs_delta_u_v",
    "completed_episode_rate",
]].sort_values("tracking_rmse_mm")

display(training_view)


### Training Plots

![Physics-informed summary bars](../../matlab_literal_env/policy_gradient_experiments/results/dyn/physics_informed_formulations_01/summary_bars.png)

![Physics-informed learning curves](../../matlab_literal_env/policy_gradient_experiments/results/dyn/physics_informed_formulations_01/learning_curves.png)

![Physics-informed actual ratio rollouts](../../matlab_literal_env/policy_gradient_experiments/results/dyn/physics_informed_formulations_01/transparency_ratio_rollouts.png)


## Force Bias 15 N Transparency Test

This diagnostic reevaluates the seven physics-informed PPO formulations with the same sinusoidal force input but with a 15 N bias: $F_h(t) = 15 + 10\sin(t)$ N. The ratio plot uses the requested force-over-velocity transparency diagnostic, with the raw trace on a symlog axis and a velocity-gated 0.5 s rolling median for monitoring.


In [ ]:
BIAS15_DIR = PHYSICS_DIR / "force_bias_15_test"
bias15_summary = pd.read_csv(BIAS15_DIR / "bias15_summary.csv")

bias15_view = bias15_summary[[
    "model_key",
    "force_amp_N",
    "force_bias_N",
    "tracking_rmse_mm",
    "transparency_rmse_w",
    "transparency_ratio_median",
    "transparency_ratio_error_rmse",
    "transparency_ratio_finite_fraction",
    "within_20pct_fraction",
    "rms_u_v",
]].sort_values("tracking_rmse_mm")

display(bias15_view)
display(Image(filename=str(BIAS15_DIR / "bias15_transparency_ratio_monitor.png")))


## Validation Ranking

This ranking uses the focused validation suite: 25 validation scenarios plus 8 empirical Bode checks per model. It includes the saved PPO baseline, all physics-informed models, and the 500k temporal stack model for context.

| Rank | Model | Family | RMS err mm | Transp RMSE W | RMS u V | Mean abs(delta u) V | Failure rate |
|---|---|---|---|---|---|---|---|
| 1 | T4_posvel_stack3_500k | temporal_stack | 4.822 | 1.2623 | 0.2934 | 0.0905 | 0.00 |
| 2 | baseline_ppoFinalModel | baseline | 6.934 | 1.2769 | 0.2110 | 0.0210 | 0.00 |
| 3 | F2_error_dot_state_reward | physics_informed | 8.054 | 1.2788 | 0.2308 | 0.0346 | 0.00 |
| 4 | F1_error_state_reward | physics_informed | 8.137 | 1.2779 | 0.2397 | 0.0280 | 0.00 |
| 5 | F3_error_ddot_state_reward | physics_informed | 11.442 | 1.1803 | 0.1978 | 0.0208 | 0.00 |
| 6 | F0_baseline | physics_informed | 20.507 | 1.1488 | 0.0838 | 0.0064 | 0.00 |
| 7 | F6_effort_plus_delta_u | physics_informed | 20.513 | 1.1489 | 0.0838 | 0.0064 | 0.00 |
| 8 | F5_accel_state_reward | physics_informed | 22.563 | 1.2171 | 0.1722 | 0.0198 | 0.00 |
| 9 | F4_accel_state | physics_informed | 23.610 | 1.2121 | 0.1562 | 0.0147 | 0.00 |

In [ ]:
validation_view = validation_summary.assign(
    rank=validation_summary["mean_rms_error_mm"].rank(method="first").astype(int),
).sort_values("mean_rms_error_mm")[[
    "rank",
    "model_key",
    "family",
    "mean_rms_error_mm",
    "mean_post_contact_rms_error_mm",
    "mean_transparency_rmse_w",
    "mean_rms_u_v",
    "mean_smoothness_abs_delta_v",
    "failure_rate",
    "mean_bode_gain_db",
    "mean_abs_bode_phase_lag_deg",
]]

display(validation_view)


## Physics-Informed Validation Only

This table isolates the physics-informed models from the validation suite. F0 is included in the CSV summary, but its validation subfolder is not currently present under `validation_test_suite_01`; the aggregate F0 metrics remain available in the summary files and top-level plots.

| Physics Rank | Model | RMS err mm | Post-contact RMS mm | Transp RMSE W | RMS u V | Mean abs(delta u) V | Validation artifacts |
|---|---|---|---|---|---|---|---|
| 1 | F2_error_dot_state_reward | 8.054 | 7.298 | 1.2788 | 0.2308 | 0.0346 | [F2_error_dot_state_reward](../../matlab_literal_env/policy_gradient_experiments/results/dyn/validation_test_suite_01/F2_error_dot_state_reward) |
| 2 | F1_error_state_reward | 8.137 | 7.408 | 1.2779 | 0.2397 | 0.0280 | [F1_error_state_reward](../../matlab_literal_env/policy_gradient_experiments/results/dyn/validation_test_suite_01/F1_error_state_reward) |
| 3 | F3_error_ddot_state_reward | 11.442 | 10.928 | 1.1803 | 0.1978 | 0.0208 | [F3_error_ddot_state_reward](../../matlab_literal_env/policy_gradient_experiments/results/dyn/validation_test_suite_01/F3_error_ddot_state_reward) |
| 4 | F0_baseline | 20.507 | 20.575 | 1.1488 | 0.0838 | 0.0064 | missing folder |
| 5 | F6_effort_plus_delta_u | 20.513 | 20.581 | 1.1489 | 0.0838 | 0.0064 | [F6_effort_plus_delta_u](../../matlab_literal_env/policy_gradient_experiments/results/dyn/validation_test_suite_01/F6_effort_plus_delta_u) |
| 6 | F5_accel_state_reward | 22.563 | 22.113 | 1.2171 | 0.1722 | 0.0198 | [F5_accel_state_reward](../../matlab_literal_env/policy_gradient_experiments/results/dyn/validation_test_suite_01/F5_accel_state_reward) |
| 7 | F4_accel_state | 23.610 | 23.476 | 1.2121 | 0.1562 | 0.0147 | [F4_accel_state](../../matlab_literal_env/policy_gradient_experiments/results/dyn/validation_test_suite_01/F4_accel_state) |

In [ ]:
physics_validation = validation_summary.query("family == 'physics_informed'").sort_values("mean_rms_error_mm")[[
    "model_key",
    "source_total_timesteps",
    "source_obs_dim",
    "mean_rms_error_mm",
    "mean_post_contact_rms_error_mm",
    "mean_transparency_rmse_w",
    "mean_transparency_ratio_mean",
    "mean_transparency_ratio_error_rmse",
    "mean_rms_u_v",
    "mean_smoothness_abs_delta_v",
    "failure_rate",
]].copy()

physics_validation["validation_folder_exists"] = physics_validation["model_key"].map(lambda k: (VALIDATION_DIR / k).exists())
display(physics_validation)


### Validation Plots

![Validation summary bars](../../matlab_literal_env/policy_gradient_experiments/results/dyn/validation_test_suite_01/validation_summary_bars.png)

![Validation group tracking heatmap](../../matlab_literal_env/policy_gradient_experiments/results/dyn/validation_test_suite_01/validation_group_tracking_heatmap.png)

![Validation Bode overlay](../../matlab_literal_env/policy_gradient_experiments/results/dyn/validation_test_suite_01/validation_bode_overlay.png)

![Validation actual ratio diagnostics](../../matlab_literal_env/policy_gradient_experiments/results/dyn/validation_test_suite_01/validation_ratio_diagnostics.png)


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4), constrained_layout=True)
physics_validation_sorted = physics_validation.sort_values("mean_rms_error_mm")

axes[0].bar(physics_validation_sorted["model_key"], physics_validation_sorted["mean_rms_error_mm"])
axes[0].set_title("Tracking RMSE")
axes[0].set_ylabel("mm")
axes[0].tick_params(axis="x", labelrotation=75)

axes[1].bar(physics_validation_sorted["model_key"], physics_validation_sorted["mean_transparency_rmse_w"])
axes[1].set_title("Transparency RMSE")
axes[1].set_ylabel("W")
axes[1].tick_params(axis="x", labelrotation=75)

axes[2].bar(physics_validation_sorted["model_key"], physics_validation_sorted["mean_rms_u_v"])
axes[2].set_title("Control Effort")
axes[2].set_ylabel("RMS u [V]")
axes[2].tick_params(axis="x", labelrotation=75)

plt.show()


## Group-Level Validation

Use this table to see where a formulation wins or loses across the validation families. Lower tracking RMSE is better.


In [ ]:
group_tracking = validation_group.pivot_table(
    index="model_key",
    columns="group",
    values="mean_rms_error_mm",
    aggfunc="mean",
)

display(group_tracking.loc[validation_view["model_key"]])


## Current Takeaways

- Best physics-informed model by validation tracking RMSE: `F2_error_dot_state_reward` at 8.054 mm.
- Best overall model in the validation suite: `T4_posvel_stack3_500k` at 4.822 mm.
- Saved PPO baseline: `baseline_ppoFinalModel` at 6.934 mm, with RMS control effort 0.2110 V.
- Within the physics-informed set, F2 and F1 are currently the strongest tracking variants; F3 reduces the W-based transparency error but gives up tracking accuracy.
- F6's added $\Delta u$ smoothness term is almost identical to F0 in this short physics-informed run, so the smoothness reward did not materially change behavior at the current training length.
- The actual `f/v` transparency ratio is included, but its enormous magnitudes confirm the expected near-zero-velocity singularity issue. Use it as a diagnostic plot/table, and use the W-based transparency RMSE for stable comparisons.


In [ ]:
artifact_rows = []
for model_key in validation_summary["model_key"]:
    model_dir = VALIDATION_DIR / model_key
    artifact_rows.append({
        "model_key": model_key,
        "folder_exists": model_dir.exists(),
        "metrics_csv": (model_dir / "focused_eval_metrics.csv").exists(),
        "summary_json": (model_dir / "focused_eval_summary.json").exists(),
        "bode_csv": (model_dir / "focused_eval_bode.csv").exists(),
        "scenario_plot_count": len(list((model_dir / "plots" / "scenarios").glob("*.png"))) if model_dir.exists() else 0,
        "empirical_bode_plot": (model_dir / "plots" / "empirical_bode.png").exists(),
    })

artifact_view = pd.DataFrame(artifact_rows)
display(artifact_view)
